Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

Author: Michal Salcik  
In this notebook, I will be trying to repurpose the original Logistic Regression model.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
#Setting a randomness seed
SEED = 314159

In [3]:
# ---- load ----
train = pd.read_csv("./data/train.csv")
score = pd.read_csv("./data/score.csv")

print("train:", train.shape)
print("score:", score.shape)

train: (15075, 31)
score: (3000, 28)


Disclaimer
--

All uses of AI are marked appropriately using #------------- notation before and after generated code  
I have also used AI to check my spelling and logical errors in writen sentences  
All mentions of generated code have used Claude Opus 5.5 (medium effort)


Exploratory Analysis
--

In [4]:
#train.describe()
#train.head()

print(set(train.columns) - set(score.columns))
#The data description only mentions missing 'claim_count' and 'total_loss_amount' in the score file, but we are also missing has_safety_program (figured by missmatch in .shape)

#Performing a must include duplicate check
#Duplicate-check function has been AI generated
#-------------------------------------------------------------------------------------------------------
def check_id_duplicates(df, id_col="policy_id", name="df"):
    """Report exact-row and ID-level duplicates. Run on RAW data (before fillna / encoding)."""
    print(f"\n=== {name}: {len(df):,} rows ===")

    n_exact = df.duplicated().sum()
    print(f"Exact duplicate rows (all columns): {n_exact:,}")

    dup_mask = df[id_col].duplicated(keep=False)
    n_dup_ids = df.loc[dup_mask, id_col].nunique()
    print(f"Unique {id_col}s: {df[id_col].nunique():,}")
    print(f"{id_col}s appearing more than once: {n_dup_ids:,} ({dup_mask.sum():,} rows)")
 
    if n_dup_ids:
        counts = df[id_col].value_counts()
        print("\nHow many times duplicated IDs appear (occurrences -> number of IDs):")
        print(counts[counts > 1].value_counts().sort_index().to_string())

        dups = df[dup_mask]
        varying = (dups.groupby(id_col).nunique(dropna=False) > 1).sum()
        varying = varying[varying > 0].sort_values(ascending=False)
        print("\nColumns that differ within duplicated IDs (number of IDs affected):")
        print(varying.to_string() if len(varying) else "  none -> pure copies")
 
    return df[dup_mask].sort_values(id_col)
#-------------------------------------------------------------------------------------------------------


#check_id_duplicates(train)
#Our train dataset includes 75 rows, the ID of which is present more than once, out of which 42 are exact duplicates and 33 differ
#All differences seem to be in the business_type column, after checking the dataset closer, I found that some labels are written in CAPITAL LETTERS, which might mean they are exact duplicates after all 

#Puttin all labels of bussines_type to lowercase
for df in (train, score):
    df["business_type"] = df["business_type"].str.strip().str.lower()
    
#check_id_duplicates(train)
#After this correction, we see that all duplicated IDs are exact duplicates, so we can move to their removal (missmatches were caused by capitalization)

#check_id_duplicates(score)
#No ID duplicates in the score dataset


#Since auto-filling all missing values with 0 creates impossible values (avg_age, years_in_business), we will take a look different missing values one-by-one and decide how to proceed
#Function for enumerating missing values in columns is AI generated and reused from a previous project
#-------------------------------------------------------------------------------------------------------
def missing_report(df, name="df"):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    report = pd.DataFrame({"missing": miss, "pct": (miss / len(df) * 100).round(2)})
    print(f"{name}: {len(report)} columns with missing values")
    return report
#-------------------------------------------------------------------------------------------------------
missing_report(train)


missing_report(score)




{'total_loss_amount', 'claim_count', 'has_safety_program'}
df: 8 columns with missing values
df: 10 columns with missing values


,missing,pct
claim_status_current_period,2700,90.0
days_to_first_claim_report,2700,90.0
claim_paid_amount_current_period,2700,90.0
first_claim_reported_date,2700,90.0
prior_loss_amount,240,8.0
late_payment_count,210,7.0
prior_year_mileage_000,180,6.0
risk_score_external,150,5.0
driver_avg_age,120,4.0
payment_frequency,90,3.0


I came up with a few sanity checks, basically checking for impossible scenarios and making sure they arent included in the dataset.  
Additional sanity checks have been suggested by AI based on the data descriptions.

In [5]:
#Sanity checks
#code created with AI asistance, based on my ideas plus a few aditional ones that i didnt realize 
#-------------------------------------------------------------------------------------------------------
pd.crosstab(train["days_to_first_claim_report"].notna(), train["claim_count"] > 0)

def sanity_checks(df):
    eff = pd.to_datetime(df["policy_effective_date"])
    exp = pd.to_datetime(df["policy_expiration_date"])
    checks = {
        "more heavy vehicles than vehicles": df["num_heavy_vehicles"] > df["vehicle_count"],
        "zero or negative vehicles":         df["vehicle_count"] <= 0,
        "zero or negative drivers":          df["driver_count"] <= 0,
        "driver avg age under 18 or over 85": (df["driver_avg_age"] < 18) | (df["driver_avg_age"] > 85),
        "negative vehicle age":              df["vehicle_avg_age"] < 0,
        "negative years in business":        df["years_in_business"] < 0,
        "negative premium or prior loss":    (df["annual_premium"] < 0) | (df["prior_loss_amount"] < 0),
        "deductible not an allowed value":   ~df["deductible"].isin([250, 500, 1000, 2500, 5000]),
        "external risk score outside 0-100": (df["risk_score_external"] < 0) | (df["risk_score_external"] > 100),
        "policy ends before it starts":      exp <= eff,
    }
    if "claim_count" in df:  # only train has outcomes
        checks["claims but zero loss"] = (df["claim_count"] > 0) & (df["total_loss_amount"] == 0)
        checks["loss but zero claims"] = (df["claim_count"] == 0) & (df["total_loss_amount"] > 0)
    return pd.Series({name: int(mask.sum()) for name, mask in checks.items()})

pd.DataFrame({"train": sanity_checks(train), "score": sanity_checks(score)})
#-------------------------------------------------------------------------------------------------------

,train,score
claims but zero loss,0,NaN
deductible not an allowed value,0,0.0
driver avg age under 18 or over 85,0,0.0
external risk score outside 0-100,0,0.0
loss but zero claims,0,NaN
more heavy vehicles than vehicles,3,0.0
negative premium or prior loss,5,0.0
negative vehicle age,0,0.0
negative years in business,0,0.0
policy ends before it starts,10,0.0


Since these are not too common, only making about 0.15% of the dataset, I will be simply dropping them

In [6]:
to_remove = (
    (train["num_heavy_vehicles"] > train["vehicle_count"])
    | (train["vehicle_count"] <= 0)
    | (train["annual_premium"] < 0) | (train["prior_loss_amount"] < 0)
    | (pd.to_datetime(train["policy_expiration_date"]) <= pd.to_datetime(train["policy_effective_date"]))
)
print("rows to drop:", to_remove.sum())
display(train[to_remove])
train = train[~to_remove].reset_index(drop=True)

rows to drop: 20


,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,risk_score_external,has_safety_program,num_heavy_vehicles,late_payment_count,claim_count,total_loss_amount,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report
906,POL-001535,INS-00309,2022-06-13,2022-01-09,2021-12-10,APD,8,8.2,4,33.0,...,56.66,1,2,0.0,0,0.00,NaN,0.00,No Claim,NaN
1180,POL-000182,INS-00882,2020-06-15,2020-02-14,2020-01-15,APD,10,4.2,5,55.2,...,47.87,0,1,0.0,0,0.00,NaN,0.00,No Claim,NaN
1783,POL-000294,INS-03322,2020-12-29,2020-11-04,2020-10-05,APD,5,0.0,12,42.5,...,22.34,0,1,0.0,0,0.00,NaN,0.00,No Claim,NaN
2571,POL-003964,INS-01029,2021-02-18,2020-09-12,2021-09-12,APD,8,9.5,6,48.0,...,NaN,0,1,0.0,0,0.00,NaN,0.00,No Claim,NaN
2789,POL-005197,INS-03747,2022-10-22,2022-07-24,2023-07-24,APD,0,4.1,9,NaN,...,16.29,1,2,2.0,0,0.00,NaN,0.00,No Claim,NaN
2959,POL-007964,INS-03704,2021-07-12,2021-04-09,2021-03-10,AL,6,5.6,7,48.9,...,57.89,1,2,1.0,0,0.00,NaN,0.00,No Claim,NaN
3044,POL-011651,INS-03642,2022-03-28,2021-11-10,2021-10-11,AL,12,5.6,5,37.2,...,4.60,0,3,0.0,0,0.00,NaN,0.00,No Claim,NaN
3763,POL-007016,INS-00084,2022-06-05,2021-12-24,2021-11-24,APD,7,7.6,4,36.6,...,68.30,0,1,3.0,2,2332.81,2022-07-25,2203.56,Open,50.0
5089,POL-003622,INS-02816,2020-08-31,2020-03-17,2020-02-16,APD,7,3.7,6,38.4,...,46.20,0,0,2.0,0,0.00,NaN,0.00,No Claim,NaN
5348,POL-001835,INS-01388,2021-03-25,2020-10-08,2021-10-08,APD,0,8.2,5,49.3,...,78.81,0,3,1.0,1,2426.76,2021-05-09,1919.01,Under Review,45.0


Using this test, we can clearly see that the days_to_first_claim_report column coresponds 1:1 with our target, meaning its previous inclusion in the features 
has caused a major leakage into our model

In [7]:
# ---- preprocessing ----

#removing duplicates
train.drop_duplicates(inplace=True, ignore_index=True)
score.drop_duplicates(inplace=True, ignore_index=True)

#for now I will not be checking on missing data, since I am planning to use a median filling approach, I will be calculating this separately after I create the train, test split
#the previous encoder also had a possible problem with encoding the two datasets separately, which would cause trouble if the datasets didnt have an exact set of posibilities in a column, thats why
#I will be wrapping these two actions (filling missing values and encoding) in a pipeline



# ---- features ----
#I removed the feature column risk_score_external as I feel the model should be of help to creating this variable instead of being fed this variable, but here i might genuinely be wrong
#I also removed 2 columns that are said to only be revealed after the instance, so they were essentially leaking data into our model
#Aditional removal of the "late_payment_count", since they are said to be counted for the year in which the data is recorded

#Right now, many columns size depends on the number of vehicles in a given fleet (for example prior_year_mileage_000 will be more for a bigger fleet), thats why I will be replacing
#them with average or % columns


v_train = train["vehicle_count"].replace(0, np.nan)
v_score = score["vehicle_count"].replace(0, np.nan)

#This transformation has been AI generated
#-------------------------------------------------------------------------------------------------------
train["drivers_per_vehicle"]          = train["driver_count"] / v_train
train["heavy_vehicle_share"]          = train["num_heavy_vehicles"] / v_train
train["mileage_per_vehicle_000"]      = train["prior_year_mileage_000"] / v_train
train["prior_apd_claims_per_vehicle"] = train["prior_apd_claim_count"] / v_train
train["prior_al_claims_per_vehicle"]  = train["prior_al_claim_count"] / v_train
train["prior_loss_per_vehicle"]       = train["prior_loss_amount"] / v_train
train["premium_per_vehicle"]          = train["annual_premium"] / v_train



score["drivers_per_vehicle"]          = score["driver_count"] / v_score
score["heavy_vehicle_share"]          = score["num_heavy_vehicles"] / v_score
score["mileage_per_vehicle_000"]      = score["prior_year_mileage_000"] / v_score
score["prior_apd_claims_per_vehicle"] = score["prior_apd_claim_count"] / v_score
score["prior_al_claims_per_vehicle"]  = score["prior_al_claim_count"] / v_score
score["prior_loss_per_vehicle"]       = score["prior_loss_amount"] / v_score
score["premium_per_vehicle"]          = score["annual_premium"] / v_score
#-------------------------------------------------------------------------------------------------------


num_cols = [
    "vehicle_count",
    "vehicle_avg_age",
    "drivers_per_vehicle",
    "driver_avg_age",
    "years_in_business",
    "mileage_per_vehicle_000",
    "prior_apd_claims_per_vehicle",
    "prior_al_claims_per_vehicle",
    "prior_loss_per_vehicle",
    "deductible",
    "coverage_limit_000",
    "premium_per_vehicle",
    "heavy_vehicle_share",
]
cat_cols = ["coverage_type", "business_type", "state", "payment_frequency"]


#AI generated transformer
#-------------------------------------------------------------------------------------------------------
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

model = Pipeline([("prep", preprocess),
                  ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
#-------------------------------------------------------------------------------------------------------





In [8]:
# ---- target ----
# using binary had-a-claim flag as target
train["target"] = (train["claim_count"] > 0).astype(int)
print("positive rate:", round(train["target"].mean(), 3))

positive rate: 0.124


In [9]:
feat_cols = num_cols + cat_cols
X = train[feat_cols]
y = train["target"]

In [10]:
# ---- fit model ----

#AI generated code for Group Shuffling
#-------------------------------------------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(X, y, groups=train["insured_id"]))
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]
#-------------------------------------------------------------------------------------------------------
#Group shuffling prevents a scenario where one company has a AL and a ADP row, one in the train and one in the test
#this could cause minor data leakage, so it is rather prevented


model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['vehicle_count','vehicle_avg_age','drivers_per_vehicle',..., 'business_type','state','payment_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

In [11]:
# ---- evaluate ----
probs = model.predict_proba(X_test)[:, 1]
preds = model.predict(X_test)

print("baseline accuracy:", round(1 - y_test.mean(), 4))
print("accuracy:", round(accuracy_score(y_test, preds), 4))
print("predicted positives:", preds.sum(), "of", len(preds))
print(confusion_matrix(y_test, preds))
print("ROC AUC:", round(roc_auc_score(y_test, probs), 4))
print("PR AUC:", round(average_precision_score(y_test, probs), 4), "| random:", round(y_test.mean(), 4))





baseline accuracy: 0.8701
accuracy: 0.8701
predicted positives: 0 of 3057
[[2660    0]
 [ 397    0]]
ROC AUC: 0.6744
PR AUC: 0.2269 | random: 0.1299


baseline accuracy: 0.8701 This would be the accuracy if the model always predicted a No Claim, so it is the baseline for our comparison  
accuracy: 0.8701
predicted positives: 0 of 3057
[[2660    0]
 [ 397    0]]
ROC AUC: 0.6744
PR AUC: 0.2269 | random: 0.1299

Even tho our model is predicting "No Claim" for every case, effectively being as good as the baseline case, we can see that it does carry real signal in the form  
of a ROC AUC score of 0.6744 (0.5 would be if predicting all "No Claim" cases) and a PR AUC score of 0.2269 compared to the random 0.1212 (about 1.7x increase)  
This shows that the score should be used as a probability, not a yes/no label  

In [12]:
# ---- score new policies ----
model.fit(X, y)
X_new = score[feat_cols]
score["risk_score"] = model.predict_proba(X_new)[:, 1]
assert score["risk_score"].notna().all() and score["policy_id"].is_unique
score[["policy_id", "risk_score"]].to_csv("predictions.csv", index=False)
print("done -- predictions.csv written")

done -- predictions.csv written


For comparison, I have decided to generate a HistGradientBoostingClassifier  
The main reason is I wanted to compare my base logistic regression model with a model more suited for working with non-linear data and see the improvements,  
or lack-there-of

In [13]:


#-------------------------------------------------------------------------------------------------------

increasing = {"prior_apd_claims_per_vehicle", "prior_al_claims_per_vehicle", "prior_loss_per_vehicle"}
monotonic = [1 if f in increasing else 0 for f in cat_cols + num_cols]

prep_gbm = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat_cols),
    ("num", "passthrough", num_cols),
])

gbm = Pipeline([
    ("prep", prep_gbm),
    ("clf", HistGradientBoostingClassifier(
        categorical_features=list(range(len(cat_cols))),
        monotonic_cst=monotonic,
        learning_rate=0.05,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=SEED,
    )),
])

gbm.fit(X_train, y_train)
gbm_probs = gbm.predict_proba(X_test)[:, 1]

print("LogReg  ROC AUC:", round(roc_auc_score(y_test, probs), 4),
      "| PR AUC:", round(average_precision_score(y_test, probs), 4))
print("GBM     ROC AUC:", round(roc_auc_score(y_test, gbm_probs), 4),
      "| PR AUC:", round(average_precision_score(y_test, gbm_probs), 4))
#-------------------------------------------------------------------------------------------------------

LogReg  ROC AUC: 0.6744 | PR AUC: 0.2269
GBM     ROC AUC: 0.6599 | PR AUC: 0.2067


LogReg  ROC AUC: 0.6744 | PR AUC: 0.2269
GBM     ROC AUC: 0.6599 | PR AUC: 0.2067 

From these values, we can see that the ROC AUC number of logistic regresion is slightly higher, meaning it might be carrying more information, but to formulate a certain conclusion, we would need to make multiple runs of the models using different train test splits and calculate their means.